# Real-Time SPA Scraper for Sandhya Fashion
Because `sandhya-fashion.in` is a Single Page Application (SPA) built with React, the products and content load dynamically via JavaScript. A simple `requests` script will only return a blank page (`<div id="root"></div>`).

To solve this instantly and get your dataset, we use **Selenium**. It launches a headless browser, waits for the products to load on the screen, and then extracts the rendered data.

In [ ]:
!pip install selenium webdriver-manager pandas beautifulsoup4

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

In [ ]:
def scrape_sandhya_fashion(url):
    # Configure Selenium to run invisibly (headless mode)
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--window-size=1920,1080")
    
    print(f"Initializing headless browser and loading {url}...")
    # Automatically download and install the correct ChromeDriver
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=chrome_options)
    
    # Load the actual dynamic webpage
    driver.get(url)
    
    print("Waiting for products to load...")
    time.sleep(8)  # Wait for React to render the products
    
    # Scroll down to ensure lazy-loaded items (like images and prices) appear
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    
    # Extract the fully rendered HTML source code
    html = driver.page_source
    driver.quit() # Close the browser to free memory
    
    print("Page rendered successfully. Extracting dataset...")
    soup = BeautifulSoup(html, 'html.parser')
    
    dataset = []
    
    # Sandhya Fashion's shop page lists products and prices.
    # We will search the HTML for product containers. A reliable way 
    # to find them without exact class names is looking for currency symbols (₹).
    
    for div in soup.find_all('div'):
        # Extract all text inside the div, separated by " | "
        text = div.get_text(separator=' | ', strip=True)
        
        # If the div contains a price (₹) and has a reasonable length, it's likely a product card
        if '₹' in text and len(text) > 15:
            # Avoid adding massive container divs that wrap the whole page
            if len(text) < 300:
                # Avoid duplicates
                if not any(text in d.get('Product_Details', '') for d in dataset):
                    
                    # Extract just the price parts for a clean column
                    prices = [chunk for chunk in text.split(' | ') if '₹' in chunk]
                    
                    dataset.append({
                        'Product_Details': text,
                        'Extracted_Price': prices[0] if prices else "N/A"
                    })
                    
    return dataset

In [ ]:
TARGET_URL = "https://www.sandhya-fashion.in/shop"

print("Starting scraping process... This will take ~10 seconds.")
data = scrape_sandhya_fashion(TARGET_URL)

if data:
    # Convert data into a structured Pandas DataFrame
    df = pd.DataFrame(data)
    
    print(f"\nSuccessfully extracted {len(df)} items!")
    display(df.head(10))
    
    # Save the dataset to a CSV file
    csv_filename = 'sandhya_dataset.csv'
    df.to_csv(csv_filename, index=False)
    print(f"\nDataset has been saved to '{csv_filename}'")
else:
    print("\nNo product data could be extracted. The site might be loading slowly.")